# Randomized Experiments, A/B Testing, and Power Lab


## Purpose

This lab simulates a randomized A/B test and estimates the treatment effect.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

n = 5000

data = pd.DataFrame({
    "unit_id": [f"unit_{i:05d}" for i in range(1, n + 1)],
    "prior_activity": rng.normal(0, 1, size=n),
    "domain_expertise": rng.normal(0, 1, size=n),
})

data["true_tau"] = (
    0.08
    + 0.04 * (data["prior_activity"] > 0)
    + 0.03 * (data["domain_expertise"] > 0)
)

data["y0"] = (
    0.30
    + 0.08 * data["prior_activity"]
    + 0.04 * data["domain_expertise"]
    + rng.normal(0, 0.05, size=n)
)

data["y1"] = data["y0"] + data["true_tau"]

data.head()

In [ ]:
data["ab_assignment"] = rng.binomial(1, 0.5, size=len(data))

data["ab_outcome"] = np.where(
    data["ab_assignment"] == 1,
    data["y1"],
    data["y0"],
)

effect_estimate = (
    data.loc[data["ab_assignment"] == 1, "ab_outcome"].mean()
    - data.loc[data["ab_assignment"] == 0, "ab_outcome"].mean()
)

balance = data.groupby("ab_assignment").agg(
    units=("unit_id", "count"),
    mean_prior_activity=("prior_activity", "mean"),
    mean_domain_expertise=("domain_expertise", "mean"),
).reset_index()

effect_estimate, balance

## Interpretation

Randomization supports exchangeability in expectation, but experiments still require balance checks, logging integrity, and valid outcome definitions.